# 05a — M1 (NeuroKit2): COMBINED detector (PTB-XL + Ningbo)

Clean rebuild of legacy 07a with M2-level rigor: gate (FDR + cross-dataset + IC95 bootstrap) -> Spearman dedup -> wide AP-vs-k -> overfit gap diagnostic -> K x config grid -> **parsimony selection (1-SE rule)** -> standardized `evaluate_standard` -> freeze. Judge: AP. Fold 10 untouched.

Feature pool = the retained **v1** morphology/delta set (~172 features, leads II/V1/V5). An enriched ~700-feature pool was tested and did **not** lift M1's ceiling — documented in `05a_DOC_m1_enrichment_experiment.ipynb`. M1 is **signal-limited**: the delta wave breaks NeuroKit delineation.

**Inputs:** `metadata_combined.csv`, raw signals via `signal_loading`, extractor `src/m1_features.py` (v1 pool, `m1_features.csv`).
**Outputs:** `models/M1_neurokit/m1_combined_*`, `reports/{figures,metrics}/`.

> Legacy 07a/b/c kept as reference; this supersedes 07a. Narrative: see `JOURNAL.md`.

### Block 5a.0: Setup

In [ ]:
# M1 (NeuroKit2 clinical + morphology) — COMBINED detector (PTB-XL + Ningbo).
# Clean rebuild of legacy 07a with M2-level rigor (wide k-search, overfit gap diagnostic, config grid,
# standardized evaluation). Extraction factored into src/m1_features.py (v1 retained morphology pool).
# Judge: AP. Fold 10 stays untouched. (An enriched ~700-feature pool was tested and REJECTED ->
# see 05a_DOC_m1_enrichment_experiment.ipynb; M1 is signal-limited.)
import os, sys, json
import numpy as np, pandas as pd
ROOT      = r'.'
PROCESSED = os.path.join(ROOT, 'data', 'processed')
MODELS    = os.path.join(ROOT, 'models', 'M1_neurokit')
FIG       = os.path.join(ROOT, 'reports', 'figures')
METRICS   = os.path.join(ROOT, 'reports', 'metrics')
for d in (MODELS, FIG, METRICS): os.makedirs(d, exist_ok=True)
sys.path.insert(0, os.path.join(ROOT, 'src'))
from m1_features import build_m1_features
from evaluation import evaluate_standard

TAG = 'combined'
FEATS_CSV = os.path.join(PROCESSED, 'm1_features.csv')
meta = pd.read_csv(os.path.join(PROCESSED, 'metadata_combined.csv'), dtype={'ecg_id': str})
wpw_fold = meta[meta.label == 1].groupby('fold').size()
print(f"metadata_combined: {len(meta)} ECGs | WPW {int((meta.label==1).sum())}")
print(f"WPW per fold: {wpw_fold.to_dict()}")
print(f"Train(1-8) {int(wpw_fold.loc[1:8].sum())} | Val(9) {int(wpw_fold.get(9,0))} | Test(10) {int(wpw_fold.get(10,0))} [UNTOUCHED]")


### Block 5a.0b: Patient-leakage check (blocking assert)

In [ ]:
# Patient-leakage check across folds (blocking assert). A patient in 2 folds = leakage
# (the model would recognize the patient, not the disease).
pat_folds = meta.groupby('patient_id')['fold'].nunique()
leaky = pat_folds[pat_folds > 1]
wpw_pat = meta[meta.label==1].patient_id.nunique()
print(f"Unique patients: {meta.patient_id.nunique()} | ECGs: {len(meta)} | in >1 fold: {len(leaky)}")
print(f"WPW: {int((meta.label==1).sum())} ECGs across {wpw_pat} unique patients")
assert len(leaky) == 0, f"PATIENT LEAK: {len(leaky)} patients in multiple folds."
print("[OK] No patient leakage across folds. Folds are clean.")


### Block 5a.1: Feature table (shared v1 extractor, guarded)

In [ ]:
# Feature table (built once via the shared module, guarded). The clinical pool 'A' is essentially
# empty for WPW (delta breaks NeuroKit delineation); the signal lives in form/delta features.
df = build_m1_features(meta, FEATS_CSV)   # ~30-60 min only if the CSV is absent
n_feat = df.shape[1] - 7
print(f"\nM1 features: {df.shape[0]} ECGs x {n_feat} features | WPW {int((df.label==1).sum())}")
for s in df.source.unique():
    sub = df[df.source == s]
    print(f"  {s}: extraction_failed  WPW {sub[sub.label==1].extraction_failed.mean()*100:.1f}%  | non-WPW {sub[sub.label==0].extraction_failed.mean()*100:.1f}%")
assert int((df.label == 1).sum()) == 142, "expected 142 WPW in the combined feature table"


### Block 5a.2: Feature selection (gate FDR + cross-dataset + dedup, guarded)

In [ ]:
# Feature selection on folds 1-8: gate = |Cohen's d|>0.3 AND p_FDR<0.05 (Benjamini-Hochberg)
# AND IC95 bootstrap of d excludes 0 AND cross-dataset coherence (separates in PTB AND Ningbo).
# Then Spearman>0.9 de-duplication (keep the strongest |d|). Guard: reload if the cached selection
# covers exactly the current feature pool (skips the ~min bootstrap gate on re-runs).
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

META = ['ecg_id','patient_id','label','fold','source','extraction_failed','n_leads_morpho_failed']
ALL_FEATS = [c for c in df.columns if c not in META] + ['n_leads_morpho_failed']
tr = df[df.fold.between(1, 8)]
SEL_CSV = os.path.join(PROCESSED, f'm1_{TAG}_selection.csv')

def cohens_d(a, b):
    a, b = a[~np.isnan(a)], b[~np.isnan(b)]
    if len(a) < 2 or len(b) < 2: return np.nan
    sp = np.sqrt(((len(a)-1)*a.var(ddof=1) + (len(b)-1)*b.var(ddof=1)) / (len(a)+len(b)-2))
    return (a.mean()-b.mean())/sp if sp > 0 else np.nan
def d_ci(a, b, n=1000, seed=42):
    rng = np.random.default_rng(seed); a, b = a[~np.isnan(a)], b[~np.isnan(b)]
    if len(a) < 2 or len(b) < 2: return (np.nan, np.nan)
    ds = [cohens_d(rng.choice(a, len(a), True), rng.choice(b, len(b), True)) for _ in range(n)]
    return tuple(np.nanpercentile(ds, [2.5, 97.5]))

if os.path.exists(SEL_CSV) and set(pd.read_csv(SEL_CSV, usecols=['feature'])['feature']) == set(ALL_FEATS):
    res = pd.read_csv(SEL_CSV)
    print(f"[guard] {os.path.basename(SEL_CSV)} reloaded ({len(res)} features) - gate skipped.")
else:
    w, n = tr[tr.label==1], tr[tr.label==0]
    ptb, nin = tr[tr.source=='ptbxl'], tr[tr.source=='ningbo']
    rows = []
    for c in ALL_FEATS:
        a, b = w[c].values.astype(float), n[c].values.astype(float)
        d = cohens_d(a, b)
        try: _, p = mannwhitneyu(a[~np.isnan(a)], b[~np.isnan(b)], alternative='two-sided')
        except Exception: p = np.nan
        lo, hi = d_ci(a, b)
        dp = cohens_d(ptb[ptb.label==1][c].values.astype(float), ptb[ptb.label==0][c].values.astype(float))
        dn = cohens_d(nin[nin.label==1][c].values.astype(float), nin[nin.label==0][c].values.astype(float))
        cross_ok = (np.isfinite(dp) and np.isfinite(dn) and np.sign(dp)==np.sign(dn) and abs(dp)>0.2 and abs(dn)>0.2)
        ci_ok = (np.isfinite(lo) and np.isfinite(hi) and (lo>0)==(hi>0))
        rows.append({'feature':c,'d':d,'d_ptb':dp,'d_nin':dn,'p_raw':p,'ci_excl0':ci_ok,'cross_ok':cross_ok})
    res = pd.DataFrame(rows); ok = res.p_raw.notna()
    res.loc[ok, 'p_FDR'] = multipletests(res.loc[ok, 'p_raw'], method='fdr_bh')[1]
    res['gate'] = (res.d.abs()>0.3) & (res.p_FDR<0.05) & res.ci_excl0 & res.cross_ok
    res = res.sort_values('d', key=lambda s: s.abs(), ascending=False).reset_index(drop=True)
    res.to_csv(SEL_CSV, index=False)
passed = res[res.gate].feature.tolist()

def dedup(feats):
    corr = tr[feats].corr(method='spearman').abs(); keep = []
    for f in feats:
        if all(corr.loc[f, k] <= 0.9 for k in keep): keep.append(f)
    return keep
dedup_list = dedup(passed)
print(f"Features tested: {len(ALL_FEATS)} | pass gate: {len(passed)} | after dedup>0.9: {len(dedup_list)}")
pd.set_option('display.float_format', lambda x: f'{x:.3f}')
print(res[['feature','d','d_ptb','d_nin','p_FDR','cross_ok','gate']].head(12).to_string(index=False))


### Block 5a.2b: Histograms of top selected features

In [ ]:
# Histograms of the top-6 selected features (WPW vs non-WPW, folds 1-8) — visual validation of separation.
import matplotlib.pyplot as plt
top6 = dedup_list[:6]
fig, ax = plt.subplots(2, 3, figsize=(15, 7))
for j, f in enumerate(top6):
    a = ax[j//3, j%3]
    wv = tr[tr.label==1][f].dropna(); nv = tr[tr.label==0][f].dropna()
    lo, hi = np.nanpercentile(tr[f], 1), np.nanpercentile(tr[f], 99)
    bins = np.linspace(lo, hi, 40)
    a.hist(nv, bins=bins, density=True, alpha=.5, color='#94a3b8', label='non-WPW')
    a.hist(wv, bins=bins, density=True, alpha=.6, color='#dc2626', label='WPW')
    a.set_title(f"{f}\n(d={res.set_index('feature').loc[f,'d']:.2f})", fontsize=9); a.legend(fontsize=7)
plt.suptitle('M1 combined — top-6 selected features (folds 1-8)'); plt.tight_layout()
plt.savefig(os.path.join(FIG, f'm1_{TAG}_histograms.png'), dpi=130, bbox_inches='tight'); plt.show()


### Block 5a.3: AP-vs-k (full pool) + fold-9 overfit guard + auto-K

In [ ]:
# AP-vs-k on the FULL deduplicated pool (TODO fix: legacy 07 capped at k=49 while AP was still rising).
# OOF AP (native folds 1-8, IC95 bootstrap) + a fold-9 probe as the overfit guard. Auto-K = smallest k
# whose AP_med >= the lower IC bound of the max (parsimonious plateau, harmonized with M2).
from sklearn.metrics import average_precision_score, roc_auc_score
from xgboost import XGBClassifier
import matplotlib.pyplot as plt

d8 = df[df.fold.between(1,8)].reset_index(drop=True); y = d8.label.values; folds = d8.fold.values
f9 = df[df.fold==9].reset_index(drop=True); yf9 = f9.label.values
FOLD_MASKS = [(folds!=h, folds==h) for h in sorted(np.unique(folds))]
spw8 = (y==0).sum()/max((y==1).sum(),1)

def make_xgb(spw=None, **kw):
    if spw is None: spw = spw8
    p = dict(n_estimators=100, max_depth=2, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
             reg_lambda=2.0, min_child_weight=3, scale_pos_weight=spw, eval_metric='aucpr',
             tree_method='hist', random_state=42, n_jobs=10)
    p.update(kw); return XGBClassifier(**p)
def oof_xgb(feats, **kw):
    X = d8[feats]; oof = np.full(len(d8), np.nan)
    for trm, vam in FOLD_MASKS:
        if y[trm].sum()==0 or y[vam].sum()==0: continue
        spw = (y[trm]==0).sum()/max((y[trm]==1).sum(),1)
        oof[vam] = make_xgb(spw, **kw).fit(X[trm], y[trm]).predict_proba(X[vam])[:,1]
    return oof
def ap_boot(yy, pp, n=1000, seed=42):
    rng = np.random.default_rng(seed); pos, neg = np.where(yy==1)[0], np.where(yy==0)[0]; a = np.empty(n)
    for i in range(n):
        idx = np.concatenate([rng.choice(pos,len(pos),True), rng.choice(neg,len(neg),True)])
        a[i] = average_precision_score(yy[idx], pp[idx])
    return np.median(a), np.percentile(a,2.5), np.percentile(a,97.5)

ks = list(range(1, len(dedup_list)+1, 2)); rows = []
for k in ks:
    oof = oof_xgb(dedup_list[:k]); m = ~np.isnan(oof)
    med, lo, hi = ap_boot(y[m], oof[m]); r = {'k':k,'AP_med':med,'IC_lo':lo,'IC_hi':hi,'AP_f9':np.nan}
    if k % 5 == 0 or k == len(dedup_list):
        mdl = make_xgb().fit(d8[dedup_list[:k]], y); r['AP_f9'] = average_precision_score(yf9, mdl.predict_proba(f9[dedup_list[:k]])[:,1])
    rows.append(r)
rk = pd.DataFrame(rows)
plt.figure(figsize=(11,5))
plt.errorbar(rk.k, rk.AP_med, yerr=[rk.AP_med-rk.IC_lo, rk.IC_hi-rk.AP_med], fmt='o-', ms=3, capsize=2, color='#2563eb', label='AP OOF folds 1-8 (IC95)')
f9p = rk.dropna(subset=['AP_f9']); plt.plot(f9p.k, f9p.AP_f9, 's-', color='#dc2626', ms=5, label='AP fold 9 (overfit guard)')
plt.axhline(0.583, ls='--', color='gray', label='M6 ref (0.583)')
plt.xlabel('k'); plt.ylabel('AP'); plt.legend(); plt.grid(alpha=.3); plt.title('M1 combined — AP vs k (full pool)')
plt.savefig(os.path.join(FIG, f'm1_{TAG}_ap_vs_k.png'), dpi=150, bbox_inches='tight'); plt.show()

kbest = rk.loc[rk.AP_med.idxmax()]; K_auto = int(rk[rk.AP_med >= kbest.IC_lo].k.min())
f9lo = f9p[f9p.k<=K_auto].AP_f9.mean(); f9hi = f9p[f9p.k>K_auto].AP_f9.mean() if (f9p.k>K_auto).any() else np.nan
print(f"Max AP OOF at k={int(kbest.k)} ({kbest.AP_med:.4f}); plateau (>= IC_lo of max) from k={K_auto}.")
print(f"fold9 AP: k<=K {f9lo:.3f} | k>K {f9hi:.3f} -> overfit {'refuted' if not np.isfinite(f9hi) or f9hi>=f9lo-0.05 else 'possible (watch)'}")
print(f"k_max (full pool) = {len(dedup_list)} | provisional K_auto = {K_auto}")


### Block 5a.4: K x config grid + parsimony selection (1-SE rule)

In [ ]:
# Overfit gap diagnostic + 2D K x config grid. Selection principle: the native-fold (anti-shuffle)
# OOF AP is the honest judge -> MAXIMIZE OOF AP among depth-2 configs (parsimony/stability), under an
# absolute gap cap <= 0.30 (as in M2; the gap guards robustness to the held-out fold). lr is the
# anti-overfit lever (M2 lesson), so low-lr configs are included. depth-3 kept only as a diagnostic.
from sklearn.metrics import average_precision_score
def diag(cfg, feats):
    X8 = d8[feats]; oof = np.full(len(d8), np.nan)
    for trm, vam in FOLD_MASKS:
        if y[trm].sum()==0 or y[vam].sum()==0: continue
        spw = (y[trm]==0).sum()/max((y[trm]==1).sum(),1)
        oof[vam] = make_xgb(spw, **cfg).fit(X8[trm], y[trm]).predict_proba(X8[vam])[:,1]
    m = ~np.isnan(oof); ao = average_precision_score(y[m], oof[m])
    mdl = make_xgb(**cfg).fit(X8, y); at = average_precision_score(y, mdl.predict_proba(X8)[:,1])
    af = average_precision_score(yf9, mdl.predict_proba(f9[feats])[:,1])
    return ao, at, at-ao, af

KS = sorted(set([max(5, K_auto//2), K_auto, min(len(dedup_list), int(K_auto*1.5)), len(dedup_list)]))
CFGS = {'d2_lr02_ne300': dict(max_depth=2, learning_rate=0.02, n_estimators=300),
        'd2_lr03_ne200': dict(max_depth=2, learning_rate=0.03, n_estimators=200),
        'd2_lr03_ne300': dict(max_depth=2, learning_rate=0.03, n_estimators=300),
        'd2_lr05_ne100': dict(max_depth=2, learning_rate=0.05, n_estimators=100),
        'd2_lr05_ne200': dict(max_depth=2, learning_rate=0.05, n_estimators=200),
        'd3_lr03_ne200': dict(max_depth=3, learning_rate=0.03, n_estimators=200)}  # depth-3 kept for diagnosis only
rows = []
for kk in KS:
    for cn, c in CFGS.items():
        ao, at, gp, af = diag(c, dedup_list[:kk])
        rows.append({'K':kk,'cfg':cn,'AP_oof':ao,'AP_train':at,'gap':gp,'AP_f9':af})
G = pd.DataFrame(rows); G.to_csv(os.path.join(PROCESSED, f'm1_{TAG}_grid2d.csv'), index=False)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')
print(G.sort_values('AP_oof', ascending=False).head(10).to_string(index=False))

GAP_CAP = 0.30
# Unified non-arbitrary selection (1-SE / bootstrap-IC rule, same as M2 06a and the K_auto plateau).
d2g = G[G.cfg.str.startswith('d2')]
gap_cap = max(GAP_CAP, float(d2g.gap.min()) + 0.05)   # 0.30 if achievable; else least-overfit depth-2 + margin
cand = d2g[d2g.gap <= gap_cap].copy()
if len(cand) == 0: cand = G[G.gap <= gap_cap].copy()
top = cand.sort_values('AP_oof', ascending=False).iloc[0]
oof_b = oof_xgb(dedup_list[:int(top.K)], **CFGS[top.cfg]); mb = ~np.isnan(oof_b)
_, ic_lo_best, _ = ap_boot(y[mb], oof_b[mb])
tied = cand[cand.AP_oof >= ic_lo_best].copy()
best = tied.sort_values(['K','AP_oof'], ascending=[True, False]).iloc[0]
K = int(best.K); FEATURES_K = dedup_list[:K]
CFG = {**CFGS[best.cfg], 'subsample':0.8, 'colsample_bytree':0.8, 'reg_lambda':2.0, 'min_child_weight':3}
print(f"\n[selection] 1-SE rule (gap_cap={gap_cap:.2f}): best depth-2 OOF AP={top.AP_oof:.3f} (K={int(top.K)} {top.cfg}), bootstrap IC_lo={ic_lo_best:.3f}")
print(f"  {len(tied)} depth-2 configs statistically tied (OOF AP >= {ic_lo_best:.3f}) -> smallest K among them:")
print(f">>> FINAL: K={K}, cfg={best.cfg} (gap {best.gap:.3f}, AP_oof {best.AP_oof:.3f}, AP_f9 {best.AP_f9:.3f}) - parsimonious, never uses fold 9")


### Block 5a.5: Final model + multi-seed + standardized evaluation

In [ ]:
# Final M1 model (chosen K + config). Raw model for SHAP/threshold; OOF on native folds for the
# threshold + Platt calibration (anti-shuffle); multi-seed stability; standardized fold-9 evaluation.
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression as _LR
from sklearn.metrics import average_precision_score, roc_auc_score

y8 = d8.label.values; X8 = d8[FEATURES_K]; X9 = f9[FEATURES_K]; y9 = f9.label.values
def make_final(**kw):
    p = dict(**CFG, scale_pos_weight=spw8, eval_metric='aucpr', tree_method='hist', random_state=42, n_jobs=10)
    p.update(kw); return XGBClassifier(**p)

xgb_raw = make_final().fit(X8, y8); score9 = xgb_raw.predict_proba(X9)[:,1]
ap_train = average_precision_score(y8, xgb_raw.predict_proba(X8)[:,1])

oof_raw = np.full(len(d8), np.nan)
for trm, vam in FOLD_MASKS:
    if y8[trm].sum()==0 or y8[vam].sum()==0: continue
    spw = (y8[trm]==0).sum()/max((y8[trm]==1).sum(),1)
    oof_raw[vam] = make_final(scale_pos_weight=spw).fit(X8[trm], y8[trm]).predict_proba(X8[vam])[:,1]
mask = ~np.isnan(oof_raw)
platt = _LR(max_iter=2000).fit(oof_raw[mask].reshape(-1,1), y8[mask])
score9_cal = platt.predict_proba(score9.reshape(-1,1))[:,1]

aps, aucs = [], []
for sd in range(10):
    mm = make_final(random_state=sd).fit(X8, y8); pp = mm.predict_proba(X9)[:,1]
    aps.append(average_precision_score(y9, pp)); aucs.append(roc_auc_score(y9, pp))
MULTISEED = dict(AP_mean=float(np.mean(aps)), AP_std=float(np.std(aps)),
                 AUC_mean=float(np.mean(aucs)), AUC_std=float(np.std(aucs)))
print(f"Multi-seed fold9: AP {np.mean(aps):.3f}+/-{np.std(aps):.3f} | AUC {np.mean(aucs):.3f}+/-{np.std(aucs):.3f}")

M1_STD = evaluate_standard(f'M1_{TAG}', y_oof=y8[mask], score_oof=oof_raw[mask], y_test=y9, score_test=score9,
    figures_dir=FIG, metrics_dir=METRICS, score_test_calibrated=score9_cal, ap_train=ap_train, multiseed=MULTISEED,
    extra={'K':K, 'xgb_params':{**CFG}, 'legacy_07a_fold9_AP':0.571})


### Block 5a.6: Freeze

In [ ]:
# Freeze M1 combined (overwrites the legacy 07a artifact names -> single canonical M1, no duplication;
# legacy numbers kept in the config for comparison). Fold 10 untouched.
import joblib
from sklearn.calibration import CalibratedClassifierCV
from datetime import datetime

oof_cal = platt.predict_proba(oof_raw.reshape(-1,1))[:,1]
final_cal = CalibratedClassifierCV(make_final(), method='sigmoid', cv=3).fit(X8, y8)
joblib.dump(xgb_raw,   os.path.join(MODELS, f'm1_{TAG}_model_raw.joblib'))    # raw (SHAP/threshold)
joblib.dump(final_cal, os.path.join(MODELS, f'm1_{TAG}_model.joblib'))        # calibrated (ensemble/Flask)
joblib.dump(platt,     os.path.join(MODELS, f'm1_{TAG}_platt.joblib'))        # Platt on OOF (added)

oof_df = d8[['ecg_id','patient_id','fold','source','label']].copy()
oof_df['proba_raw'] = oof_raw; oof_df['proba_cal'] = oof_cal
oof_df.to_csv(os.path.join(PROCESSED, f'm1_{TAG}_oof.csv'), index=False)
pd.Series(FEATURES_K, name='feature').to_csv(os.path.join(MODELS, f'm1_{TAG}_features.csv'), index=False)

config = {
    'model': f'M1_neurokit_{TAG}',
    'train': 'PTB-XL + Ningbo, folds 1-8',
    'representation': 'NeuroKit2 clinical + morphology/delta (leads II/V1/V5)',
    'K': int(K), 'features': FEATURES_K,
    'xgb_params': {**CFG, 'scale_pos_weight': float(spw8)},
    'calibration': 'Platt (LogisticRegression) on OOF native folds (anti-shuffle)',
    'selection': 'gate |d|>0.3 & p_FDR<0.05 (BH) & IC95 excl 0 & cross-dataset; dedup Spearman>0.9; '
                 'K & config via wide AP-vs-k + overfit gap + 2D K x config grid',
    'metrics_standard_fold9': M1_STD, 'multiseed': MULTISEED,
    'legacy_07a': {'K': 35, 'AP_fold9': 0.571, 'AUC_fold9': 0.968,
                   'note': 're-derived with M2-level rigor; this notebook (05a) supersedes 07a'},
    'fold10': 'UNTOUCHED — reserved for the final ensemble evaluation',
    'date_frozen': datetime.now().strftime('%Y-%m-%d %H:%M'),
}
json.dump(config, open(os.path.join(MODELS, f'm1_{TAG}_config.json'), 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
print(f"M1 {TAG} FROZEN | K={K} | fold9 AP {M1_STD['AP']:.3f} (legacy 0.571) | AUC {M1_STD['AUC']:.3f} "
      f"| F1 {M1_STD['confusion_at_threshold']['f1']:.3f}")
